# Agent的高级用法

## 1、设置agent的名称

使用字段name来进行设置

举例：

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 以init_chat_model为例
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)



agent = create_agent(
    model=model,
    name="agent01"
)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})


# rprint(response)

for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题
================================ Human Message =================================

100 + 20 * 3 = ？
================================== Ai Message ==================================
Name: agent01

先算乘法，再算加法：

- \(20 \times 3 = 60\)
- \(100 + 60 = 160\)

所以答案是 **160**。


## 2、系统提示词的设置

使用system_prompt参数进行设置，可以是str,也可以是SystemMessage

举例1：

In [3]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)


# 1.导入模型
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.导入工具
web_search = TavilySearch(max_results=2)

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='4cbf4503-6a11-4203-b1ec-c2486d0df3c5'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 45,
                    'prompt_tokens': 1304,
                    'total_tokens': 1349,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 5,
                        'engine_ttft_ms': 69,
                        'engine_ttlt_ms': 308,
                        'pre_inference_ms': 115,
                        'service_tbt_ms': 5,
                        'service_ttft_ms': 452,
                        'service_ttlt_ms': 685,
                        'total_duration_ms': 579,
                        'user_visible_ttft_ms': 336
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-DmhgtZz6yHXJhtIilb9FpxccnmiMr',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e8e12-dd78-73f3-9831-7c2eaf693819-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {
                        'query': '2026 FIFA World Cup host countries official',
                        'include_domains': ['fifa.com'],
                        'search_depth': 'basic',
                        'topic': 'general'
                    },
                    'id': 'call_6vKGMBbkjSa8TeWtuUI2ZSkH',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1304,
                'output_tokens': 45,
                'total_tokens': 1349,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"query": "2026 FIFA World Cup host countries official", "follow_up_questions": null, 
"answer": null, "images": [], "results": [{"url": 
"https://www.fifa.com/en/articles/world-cup-2026-host-city-houston-guide", "title": "Houston | Host City Guide | 
FIFA World Cup 2026™", "content": "# Houston | Host City Guide | FIFA World Cup 2026™. # Houston. Host City: 
Houston Host City: Houston. A taste of what’s to come from this FIFA World Cup 26™ Host City.A taste of what’s to 
come from this FIFA World Cup 26™ Host City. Learn more about the host city, which will stage seven matches at FIFA
World Cup 2026™. *   **The FIFA World Cup is heading to Houston in 2026**. As one of the most diverse destinations 
in America, the city will utilize the power of the FIFA World Cup™ to bring communities together to showcase the 
city to the world, providing a once-in-a lifetime experience for Houstonians, and leveraging the spirit of the 
event to leave a lasting legacy to football in the United States. #### **Explore hospitality packages for Houston’s
FIFA World Cup 2026™ matches****Explore hospitality packages for Houston’s FIFA World Cup 2026™ matches**.", 
"score": 0.99983394, "raw_content": null}, {"url": 
"https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/host-cities", "title": "Host Cou

举例2：

In [4]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"


# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)

rprint(response)
# print(response["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='10加上20再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='34e2456e-56f0-4c26-9217-85db417e3b47'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 56,
                    'prompt_tokens': 158,
                    'total_tokens': 214,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 35,
                        'engine_ttlt_ms': 228,
                        'pre_inference_ms': 70,
                        'service_tbt_ms': 4,
                        'service_ttft_ms': 270,
                        'service_ttlt_ms': 455,
                        'total_duration_ms': 397,
                        'user_visible_ttft_ms': 200
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-DmhifgG1ssb0PfAZuGue1S32r19Ye',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e8e14-8bbb-7201-9c81-8f986d7af29a-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 10, 'b': 20},
                    'id': 'call_QAukl8rGvhJmiDWnHvAW9Lid',
                    'type': 'tool_call'
                },
                {
                    'name': 'add_numbers',
                    'args': {'a': 30, 'b': 0},
                    'id': 'call_zAH6sRjbIXONsvcedT6le9Vl',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 158,
                'output_tokens': 56,
                'total_tokens': 214,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='ca45b1df-504d-4d0e-ab2e-00d75a8a64dc',
            tool_call_id='call_QAukl8rGvhJmiDWnHvAW9Lid'
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='f1e44533-7f05-421f-a35e-0db88d454dc7',
            tool_call_id='call_zAH6sRjbIXONsvcedT6le9Vl'
        ),
        AIMessage(
            content='10加上20再加上30是 **60**。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 17,
                    'prompt_tokens': 238,
                    'total_tokens': 255,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 37,
                        'engine_ttlt_ms': 99,
                        'pre_inference_ms': 137,
                        'service_tbt_ms': 3,
                        'servi

举例3：

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1

    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"


messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，杭州今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Sv4KotTGb5c1aiUOhFsCzlao)
 Call ID: call_Sv4KotTGb5c1aiUOhFsCzlao
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_MNlT2p9GrmF9gHaKglDY1flh)
 Call ID: call_MNlT2p9GrmF9gHaKglDY1flh
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_dOyfcXs0f9K6rDgMI1hiMYLm)
 Call ID: call_dOyfcXs0f9K6rDgMI1hiMYLm
  Args:
    city: 杭州
================================= To

## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略：

In [3]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ProviderStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='c76c17a2-3e32-4eb4-bdbf-411d8ac29cea'
        ),
        AIMessage(
            content='{"name":"小明","email":"shkstart@atguigu.com","phone":"13012341234"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 242,
                    'total_tokens': 273,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 5,
                        'engine_ttft_ms': 30,
                        'engine_ttlt_ms': 183,
                        'pre_inference_ms': 71,
                        'service_tbt_ms': 5,
                        'service_ttft_ms': 254,
                        'service_ttlt_ms': 392,
                        'total_duration_ms': 333,
                        'user_visible_ttft_ms': 183
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-DmtjPEaEDe1aUSY1RYSlCW4YTciqH',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019e90d5-18fe-76d3-af7d-68d77aea40a9-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 242,
                'output_tokens': 31,
                'total_tokens': 273,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

### 3.2 ToolStrategy策略：

In [5]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ToolStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)

# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='3feca91d-5782-4ed7-b298-83fbccafbf42'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 36,
                    'prompt_tokens': 170,
                    'total_tokens': 206,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 33,
                        'engine_ttlt_ms': 134,
                        'pre_inference_ms': 74,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 254,
                        'service_ttlt_ms': 354,
                        'total_duration_ms': 297,
                        'user_visible_ttft_ms': 180
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-DmtnSxyNdNiVaXjWKJPslD8mBkgYU',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e90d8-f129-7d33-be96-94d7902cfa46-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_nwXQR1YHUJDp2eARMiilsNTR',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 170,
                'output_tokens': 36,
                'total_tokens': 206,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='59bd58a0-b0e6-407a-8b79-fced3343e2c1',
            tool_call_id='call_nwXQR1YHUJDp2eARMiilsNTR'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

### 3.3 AutoStrategy / Type策略：

In [7]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=AutoStrategy(ContractInfo)
    # response_format=ContractInfo  # 不推荐大家使用
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)

# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='2b329df9-a2eb-4a0e-9997-53d7595771cd'
        ),
        AIMessage(
            content='{"name":"小明","email":"shkstart@atguigu.com","phone":"13012341234"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 242,
                    'total_tokens': 273,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 49,
                        'engine_ttlt_ms': 184,
                        'pre_inference_ms': 81,
                        'service_tbt_ms': 5,
                        'service_ttft_ms': 304,
                        'service_ttlt_ms': 432,
                        'total_duration_ms': 364,
                        'user_visible_ttft_ms': 223
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dmtq1yMoKsuXffvmq8rlhB1bIdDKh',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019e90db-5c15-7c00-b95d-165795280692-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 242,
                'output_tokens': 31,
                'total_tokens': 273,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}